# Google Colab Training Pipeline

**Objective:** Train deepfake detection models on Google Colab with GPU acceleration.

**Drive Setup:**
- **Source Drive (Datasets):** Primary Google Drive account containing FaceForensics++ and Celeb-DF datasets
- **Target Drive (Results):** Secondary Google Drive account for saving trained models and logs

**Research Traceability:**
- Research Objective: Reproducible model training with GPU acceleration
- Methodology: Transfer learning with XceptionNet and EfficientNet-B0
- Implementation: notebooks/colab_training.ipynb

## 1. Environment Setup

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be slow.")
    print("Go to Runtime > Change runtime type > GPU")

In [ ]:
# Install dependencies
!pip install -q torch torchvision torchaudio
!pip install -q scikit-learn matplotlib seaborn pillow
!pip install -q pyyaml python-dotenv tqdm
!pip install -q albumentations

print("Dependencies installed")

In [ ]:
# Clone repository (if not already present)
import os
from pathlib import Path

REPO_DIR = Path("/content/deepfake-social-media-detector")
if not REPO_DIR.exists():
    !git clone https://github.com/YOUR_USERNAME/deepfake-social-media-detector.git /content/deepfake-social-media-detector
    print("Repository cloned")
else:
    print("Repository already exists")

# Change to repo directory
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## 2. Google Drive Mounting

Mount both Google Drive accounts:
- **Source Drive:** Contains datasets (FaceForensics++, Celeb-DF)
- **Target Drive:** For saving trained models and experiment logs

In [ ]:
# Mount Source Drive (Datasets)
from google.colab import drive

SOURCE_DRIVE_PATH = "/content/drive_source"
os.makedirs(SOURCE_DRIVE_PATH, exist_ok=True)

print("Mounting Source Drive (Datasets)...")
print("When prompted, select the Google account with your datasets")
drive.mount(SOURCE_DRIVE_PATH, force_remount=True)

# Update this path to your dataset location on Source Drive
SOURCE_DATASET_DIR = Path(SOURCE_DRIVE_PATH) / "My Drive" / "datasets"
print(f"Source dataset directory: {SOURCE_DATASET_DIR}")

In [ ]:
# Mount Target Drive (Results)
TARGET_DRIVE_PATH = "/content/drive_target"
os.makedirs(TARGET_DRIVE_PATH, exist_ok=True)

print("Mounting Target Drive (Results)...")
print("When prompted, select the secondary Google account for results")
# Note: Colab only supports one Drive mount at a time
# We'll use symlinks or copy files between drives
print("\nNOTE: Colab supports only one Drive mount at a time.")
print("We'll save results locally first, then upload to Target Drive.")

# For now, we'll use the same mount for both
# Later we'll upload results to the second drive
TARGET_OUTPUT_DIR = Path("/content/outputs")
os.makedirs(TARGET_OUTPUT_DIR, exist_ok=True)
print(f"Local output directory: {TARGET_OUTPUT_DIR}")

## 3. Dataset Preparation

In [ ]:
# Check dataset availability
import subprocess

def check_dataset_structure(dataset_dir: Path) -> dict:
    """Check if dataset directory has expected structure."""
    info = {}
    
    if not dataset_dir.exists():
        info["exists"] = False
        return info
    
    info["exists"] = True
    info["contents"] = list(dataset_dir.iterdir())
    
    # Check for FaceForensics++ structure
    ff_dirs = ["real", "fake", "manipulated_sequences", "original_sequences"]
    info["faceforensics"] = any((dataset_dir / d).exists() for d in ff_dirs)
    
    # Check for Celeb-DF structure
    celeb_dirs = ["Celeb-real", "Celeb-synthesis", "YouTube-real"]
    info["celebdf"] = any((dataset_dir / d).exists() for d in celeb_dirs)
    
    return info

dataset_info = check_dataset_structure(SOURCE_DATASET_DIR)
print(f"Dataset directory exists: {dataset_info.get('exists', False)}")
if dataset_info.get("exists"):
    print(f"FaceForensics++ detected: {dataset_info.get('faceforensics', False)}")
    print(f"Celeb-DF detected: {dataset_info.get('celebdf', False)}")
    print("\nContents:")
    for item in dataset_info.get("contents", []):
        print(f"  {item.name}")
else:
    print("\nDataset not found. Please:")
    print("1. Upload datasets to your Source Drive")
    print("2. Update SOURCE_DATASET_DIR path above")

In [ ]:
# Copy datasets to local storage for faster access
LOCAL_DATA_DIR = Path("/content/data")
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

if dataset_info.get("exists"):
    print("Copying datasets to local storage (faster access)...")
    !rsync -av --progress "{SOURCE_DATASET_DIR}/" "{LOCAL_DATA_DIR}/"
    print(f"\nLocal data directory: {LOCAL_DATA_DIR}")
    print(f"Contents:")
    !ls -la "{LOCAL_DATA_DIR}"

## 4. Data Preprocessing

Extract frames, detect faces, and prepare datasets for training.

In [ ]:
# Run preprocessing pipeline
import sys
sys.path.insert(0, str(REPO_DIR))

from src.preprocessing.frame_extractor import FrameExtractor
from src.preprocessing.face_cropper import FaceCropper
from src.preprocessing.image_resizer import ImageResizer
from src.preprocessing.normalizer import ImageNormalizer

print("Preprocessing pipeline initialized")
print("Run the following command to preprocess your dataset:")
print(f"  python scripts/preprocess.sh --input {LOCAL_DATA_DIR} --output /content/data/processed")

## 5. Training Configuration

In [ ]:
# Training configuration
TRAINING_CONFIG = {
    # Model settings
    "model": "xception",  # xception or efficientnet_b0
    "num_classes": 2,
    "pretrained": True,
    
    # Training hyperparameters
    "learning_rate": 0.001,
    "batch_size": 32,
    "epochs": 50,
    "weight_decay": 0.0001,
    "label_smoothing": 0.1,
    
    # Early stopping
    "early_stopping_patience": 10,
    "min_delta": 0.001,
    
    # Data settings
    "image_size": 299,
    "num_workers": 2,
    
    # Reproducibility
    "seed": 42,
    
    # Output settings
    "experiment_name": "colab_xception_baseline",
}

print("Training Configuration:")
for k, v in TRAINING_CONFIG.items():
    print(f"  {k}: {v}")

## 6. Model Initialization

In [ ]:
# Initialize model
from src.models.model_factory import create_model
from src.config.settings import ModelConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_config = ModelConfig(
    name=TRAINING_CONFIG["model"],
    num_classes=TRAINING_CONFIG["num_classes"],
    pretrained=TRAINING_CONFIG["pretrained"],
)

model = create_model(model_config, device)

# Print model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model: {TRAINING_CONFIG['model']}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Device: {device}")

## 7. Training Loop

In [ ]:
# Training function
import time
import json
from datetime import datetime

def train_model(model, train_loader, val_loader, config, device):
    """Train the model with early stopping and checkpointing."""
    
    # Setup
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.1, patience=5
    )
    
    loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=config["label_smoothing"])
    
    # Callbacks
    from src.training.callbacks import EarlyStopping, ModelCheckpoint
    
    checkpoint_dir = TARGET_OUTPUT_DIR / "checkpoints"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    
    callbacks = [
        EarlyStopping(
            patience=config["early_stopping_patience"],
            monitor="val_loss",
            min_delta=config["min_delta"],
        ),
        ModelCheckpoint(
            save_dir=str(checkpoint_dir),
            monitor="val_loss",
            save_best=True,
            save_last=True,
        ),
    ]
    
    # Trainer
    from src.training.trainer import Trainer
    
    trainer = Trainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        loss_fn=loss_fn,
        device=device,
        num_classes=config["num_classes"],
        callbacks=callbacks,
    )
    
    # Train
    print(f"\nStarting training for {config['epochs']} epochs...")
    start_time = time.time()
    
    history = trainer.train(num_epochs=config["epochs"])
    
    training_time = time.time() - start_time
    print(f"\nTraining completed in {training_time/60:.1f} minutes")
    
    # Save history
    history_path = TARGET_OUTPUT_DIR / "training_history.json"
    with open(history_path, "w") as f:
        json.dump(history, f, indent=2)
    print(f"Training history saved to {history_path}")
    
    return history, trainer

print("Training function defined")

In [ ]:
# Run training (uncomment when dataset is ready)
# history, trainer = train_model(model, train_loader, val_loader, TRAINING_CONFIG, device)

## 8. Training Visualization

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt
import json

def plot_training_curves(history_path: str):
    """Plot training and validation curves."""
    if not Path(history_path).exists():
        print("No training history found")
        return
    
    with open(history_path) as f:
        history = json.load(f)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Loss curve
    axes[0].plot(history["train_loss"], label="Train")
    axes[0].plot(history["val_loss"], label="Validation")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    
    # Accuracy curve
    axes[1].plot(history["train_acc"], label="Train")
    axes[1].plot(history["val_acc"], label="Validation")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig(TARGET_OUTPUT_DIR / "training_curves.png", dpi=150)
    plt.show()
    print(f"Plot saved to {TARGET_OUTPUT_DIR / 'training_curves.png'}")

# Uncomment after training:
# plot_training_curves(str(TARGET_OUTPUT_DIR / "training_history.json"))

## 9. Save Results to Target Drive

Upload trained models and logs to the secondary Google Drive account.

In [ ]:
# Upload results to Google Drive
import shutil
from datetime import datetime

def upload_results_to_drive(local_output_dir: Path, drive_path: str):
    """Upload results to Google Drive."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    drive_upload_dir = Path(drive_path) / "My Drive" / "deepfake_results" / timestamp
    
    print(f"Uploading results to: {drive_upload_dir}")
    
    # Create directory
    drive_upload_dir.mkdir(parents=True, exist_ok=True)
    
    # Copy files
    for item in local_output_dir.rglob("*"):
        if item.is_file():
            rel_path = item.relative_to(local_output_dir)
            dest = drive_upload_dir / rel_path
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(item, dest)
            print(f"  Uploaded: {rel_path}")
    
    print(f"\nResults uploaded to: {drive_upload_dir}")
    return drive_upload_dir

# Uncomment after training:
# upload_results_to_drive(TARGET_OUTPUT_DIR, SOURCE_DRIVE_PATH)

In [ ]:
# Alternative: Download results as zip
def download_results_as_zip(local_output_dir: Path):
    """Create zip file of results for download."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    zip_path = f"/content/deepfake_results_{timestamp}"
    
    print(f"Creating zip file: {zip_path}.zip")
    !zip -r "{zip_path}.zip" "{local_output_dir}"
    
    print(f"\nDownload the zip file from:")
    print(f"  /content/deepfake_results_{timestamp}.zip")
    print(f"\nThen upload to your Target Drive manually.")
    
    return f"{zip_path}.zip"

# Uncomment after training:
# download_results_as_zip(TARGET_OUTPUT_DIR)

## 10. Evaluation

Run evaluation on test set after training.

In [ ]:
# Evaluation function
def evaluate_model(model, test_loader, device):
    """Evaluate model on test set."""
    model.eval()
    correct = 0
    total = 0
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    accuracy = correct / total
    print(f"Test Accuracy: {accuracy:.4f}")
    
    return accuracy, all_preds, all_labels

print("Evaluation function defined")
print("Uncomment below to run evaluation after training:")
# accuracy, preds, labels = evaluate_model(model, test_loader, device)

## Summary

### Workflow:
1. **Setup:** Mount Source Drive with datasets
2. **Preprocess:** Extract frames, detect faces, prepare data
3. **Train:** Run training with GPU acceleration
4. **Evaluate:** Test model performance
5. **Upload:** Save results to Target Drive

### Important Notes:
- **GPU Required:** Go to Runtime > Change runtime type > GPU
- **Dataset Location:** Update `SOURCE_DATASET_DIR` to match your Drive structure
- **Results:** Saved locally first, then uploaded to Target Drive
- **Reproducibility:** Seed is fixed at 42 for all experiments